In [ ]:
import tskit
import numpy as np
import gaiapy as gp
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from tqdm import tqdm
import time
import multiprocessing as mp

: 

In [ ]:
def locations(ts):
    nodes = ts.nodes()
    locs_array = []
    for node in nodes:
        if node.individual != -1:
            ind = ts.individual(node.individual)
            x = ind.location[0]
            y = ind.location[1]
            is_sample = node.is_sample()
            locs_array.append(node.id)
            locs_array.append(is_sample)
            locs_array.append(x)
            locs_array.append(y)
        
    locs_array = np.array(locs_array)
    locs = locs_array.reshape(-1, 4)
    return locs


#function that takes the tree sequence and finds the unary nodes in the tree sequence
# returns a list of all the unary indices by node id 
def findUnary(ts):
    unary_nodes = np.zeros(ts.num_nodes) # binary vector specifying if a node is unary or not anywhere on the tree sequence
    for tree in ts.trees():
        num_children = tree.num_children_array[:ts.num_nodes]
        is_unary = num_children == 1
        for i, condition in enumerate(is_unary):
            if is_unary[i] == True:
                unary_nodes[i] = 1
    mask = unary_nodes == 1
    #mask, unary_nodes[mask]
    unary_list = np.where(mask)
    unary_indices = unary_list[0]
    return unary_indices

# finds the distinct children of unary nodes and returns their indices 
def find_children(unary_indices, node_stats_df):
    children = set()
    for node_id in unary_indices:
        n = node_stats_df.loc[node_id, 'distinct_children']
        if n is None or len(n) == 0:
            continue
        n = n[n != -1]
        children.update(n)
    return np.array(list(children))

def find_parents(unary_indices, node_stats_df):
    parents = set()
    for node_id in unary_indices:
        n = node_stats_df.loc[node_id, 'distinct_parents']
        if n is None or len(n) == 0:
            continue
        n = n[n != -1]
        parents.update(n)
    return np.array(list(parents))


def node_spans(ts, include_missing=False):
    """
    Returns the array of "node spans", i.e., the `j`th entry gives
    the total span over which node `j` is in the tree sequence.
    Sample nodes that are isolated are "missing data"; inclusion
    of these spans are controlled by `include_missing`. (If
    `include_missing` is `True` then the span of each sample is
    always equal to the sequence length.)

    :param bool include_missing: Whether to include spans of nodes
        on which they have missing data.
    """
    child_spans = np.bincount(
        ts.edges_child,
        weights=ts.edges_right - ts.edges_left,
        minlength=ts.num_nodes,
    )
    for t in ts.trees():
        span = t.span
        for r in t.roots:
            # do this check to exempt 'missing data'
            if include_missing or (t.num_children(r) > 0):
                child_spans[r] += span
    return child_spans

def get_span_stats(ts, ets):
    node_map = {}
    added_span = np.zeros(ets.num_nodes)
    wrong_added_span = np.zeros(ets.num_nodes)
    for n in ts.nodes():

        slim_id = n.metadata["slim_id"]
        assert slim_id not in node_map
        node_map[slim_id] = n.id

    for interval, t, et in ts.coiterate(ets):
        interval_length = interval[1] - interval[0]
        t_nodes = list(t.nodes())
        #et_nodes = list(et.nodes())
        for n in et.nodes():
            # print("et nodes", et_nodes)
            if et.num_children(n) == 1:
                added_span[n] += interval_length
            #on = node_map[n]
            # for x in on:
            node = ets.node(n)
            on = node_map[node.metadata["slim_id"]]
            if on not in t_nodes:
                assert et.num_children(n) == 1, print(interval, n, et.num_children(n), et.time(n))
                wrong_added_span[n] += interval_length
                #print("added ", interval_length, " to wrong_added_span")
    # assert not np.array_equal(added_span, wrong_added_span)
    return added_span, wrong_added_span

def get_node_stats(ts):
    node_ids = np.array([n.id for n in ts.nodes()])
    is_sample = np.isin(node_ids, ts.samples()).astype(int)
    
    children_set = {}
    for nid in node_ids:
        children_set[nid] = set()

    parents_set = {}
    for nid in node_ids:
        parents_set[nid] = set()

    is_root = {}
    for nid in node_ids:
        is_root[nid] = 0

    for tree in tqdm(ts.trees()): 
        for nid in node_ids:
            if tree.is_root(nid):
                is_root[nid] = 1
            children_set[nid].update(tree.children(nid))
            p = tree.parent(nid)
            if p != -1:
                parents_set[nid].add(p)

    distinct_children = []
    for nid in node_ids:
        as_list = list(children_set[nid])
        as_array = np.array(as_list)
        distinct_children.append(as_array)

    distinct_parents = []
    for nid in node_ids:
        as_list = list(parents_set[nid])
        as_array = np.array(as_list)
        distinct_parents.append(as_array)

    data_dict = pd.DataFrame({
        'id': node_ids,
        'num_children': [len(children_set[nid]) for nid in node_ids],
        'distinct_children': distinct_children,
        'distinct_parents': distinct_parents,
        'num_parents': [len(parents_set[nid])  for nid in node_ids],
        'is_sample': is_sample,
        'is_root': np.array([is_root[nid] for nid in node_ids])
    })

    return data_dict

In [ ]:
def worker_get_node_stats(ts_path, result_queue):
    ts = tskit.load(ts_path)
    result_queue.put('node_stats', get_node_stats(ts))

def worker_get_span_stats(ts_path, ets_path, result_queue):
    ts = tskit.load(ts_path)
    ets = tskit.load(ets_path)
    result_queue.put(('span_stats', get_span_stats(ts, ets)))

def worker_get_node_spans(sts_path, ets_path, result_queue):
    sts = tskit.load(sts_path)
    ets = tskit.load(ets_path)
    result_queue.put(('node_spans', (node_spans(sts), node_spans(ets))))

def worker_gaia(sts_path, ets_path, sample_locations, result_queue):

    sts = tskit.load(sts_path)
    ets = tskit.load(ets_path)

    sts_mpr = gp.quadratic_mpr(sts, sample_locations)
    sts_map_x = gp.quadratic_mpr_minimize(sts_mpr)
    ets_mpr = gp.quadratic_mpr(ets, sample_locations)
    ets_map_x = gp.quadratic_mpr_minimize(ets_mpr)

    result_queue.put(('gaia', (sts_map_x, ets_map_x)))


In [ ]:
def getAccOut(ts_path, out_prefix, sigma, rep):
    ts = tskit.load(ts_path)
    sts = ts.simplify()
    ets = sts.extend_haplotypes()

    sts_path = f"{out_prefix}_sts_temp.trees"
    ets_path  = f"{out_prefix}_ets_temp.trees"
    sts.dump(sts_path)
    ets.dump(ets_path)

    sts_num_trees = sts.num_trees
    ets_num_trees  = ets.num_trees
    sts_num_edges = sts.num_edges
    ets_num_edges  = ets.num_edges

    locs = locations(ets)
    sample_locations = locs[locs[:, 1] == 1][:, [0, 2, 3]]
    sample_centroid = np.mean(sample_locations[:, 1:], axis=0)

    print("done with getting stuff before attempting multiprocessing")

    result_queue = mp.Queue()

    processes = [
        mp.Process(target=worker_get_node_stats, args=(ts_path, result_queue)),
        mp.Process(target=worker_get_span_stats, args=(ts_path, ets_path, result_queue)),
        mp.Process(target=worker_get_node_spans, args=(sts_path, ets_path, result_queue)),
        mp.Process(target=worker_gaia, args=(sts_path, ets_path, sample_locations, result_queue)),
    ]

    for p in processes:
        p.start()

    # Collect all 4 results
    results = {}
    for i in processes:
        key, val = result_queue.get()
        results[key] = val

    for p in processes:
        p.join()

    print("done multi processing")

    node_stats_df = results['node_stats']
    total_added_span, wrongly_added_span = results['span_stats']
    sts_spans, ets_spans = results['span_stats']
    sts_map_x, ets_map_x = results['gaia']

    os.remove(sts_path)
    os.remove(ets_path)

    print("done like getting results from multiprocessing")

    unary_indices = findUnary(ets)
    children = find_children(unary_indices, node_stats_df)
    parents = find_parents(unary_indices,  node_stats_df)


    is_unary = np.isin(locs[:, 0], unary_indices)
    is_ancestor = np.isin(locs[:, 0], unary_indices)
    is_child = np.isin(locs[:,0], children)
    is_parent = np.isin(locs[:,0], parents)
    unary_nodes = is_unary & is_ancestor 
    child_nodes = is_child
    parent_nodes = is_parent

    unary_locations = locs[unary_nodes][:, [0, 2, 3]]
    child_locations = locs[child_nodes][:, [0, 2, 3]]
    parent_locations = locs[parent_nodes][:, [0, 2, 3]]

    unary_sts_e = np.sqrt(np.sum((sts_map_x[unary_nodes] - unary_locations[:, 1:2])**2, axis=1)) / np.max(pdist(sample_locations[:, 1:2]))
    unary_ets_e = np.sqrt(np.sum((ets_map_x[unary_nodes] - unary_locations[:, 1:2])**2, axis=1)) / np.max(pdist(sample_locations[:, 1:2]))

    child_sts_e = np.sqrt(np.sum((sts_map_x[child_nodes] - child_locations[:, 1:2])**2, axis=1)) / np.max(pdist(sample_locations[:, 1:2]))
    child_ets_e = np.sqrt(np.sum((ets_map_x[child_nodes] - child_locations[:, 1:2])**2, axis=1)) / np.max(pdist(sample_locations[:, 1:2]))

    parent_sts_e = np.sqrt(np.sum((sts_map_x[parent_nodes] - parent_locations[:, 1:2])**2, axis=1)) / np.max(pdist(sample_locations[:, 1:2]))
    parent_ets_e = np.sqrt(np.sum((ets_map_x[parent_nodes] - parent_locations[:, 1:2])**2, axis=1)) / np.max(pdist(sample_locations[:, 1:2]))
  
    unary_dist_from_sample_centroid0 = np.sqrt(np.sum((unary_locations[:, 1:3] - sample_centroid)**2, axis=1))
    child_dist_from_sample_centroid0 = np.sqrt(np.sum((child_locations[:, 1:3] - sample_centroid)**2, axis=1))
    parent_dist_from_sample_centroid0 = np.sqrt(np.sum((parent_locations[:, 1:3] - sample_centroid)**2, axis=1))

    unary_sts_dist_from_sample_centroid = np.sqrt(np.sum((sts_map_x[unary_nodes] - sample_centroid)**2, axis=1))
    unary_ets_dist_from_sample_centroid = np.sqrt(np.sum((ets_map_x[unary_nodes] - sample_centroid)**2, axis=1))

    child_sts_dist_from_sample_centroid = np.sqrt(np.sum((sts_map_x[child_nodes] - sample_centroid)**2, axis=1))
    child_ets_dist_from_sample_centroid = np.sqrt(np.sum((ets_map_x[child_nodes] - sample_centroid)**2, axis=1))

    parent_sts_dist_from_sample_centroid = np.sqrt(np.sum((sts_map_x[parent_nodes] - sample_centroid)**2, axis=1))
    parent_ets_dist_from_sample_centroid = np.sqrt(np.sum((ets_map_x[parent_nodes] - sample_centroid)**2, axis=1))

    unary_node_ids = locs[unary_nodes, 0]
    unary_node_times = ets.nodes_time[unary_nodes]

    child_node_ids = locs[child_nodes, 0]
    child_node_times = ets.nodes_time[child_nodes]

    parent_node_ids = locs[parent_nodes, 0]
    parent_node_times = ets.nodes_time[parent_nodes]

    sts_spans = sts_spans[unary_nodes]
    ets_spans = ets_spans[unary_nodes]

    total_added_span = total_added_span[unary_nodes]
    wrong_added_span = wrongly_added_span[unary_nodes]

    unary_df = pd.DataFrame({
        'unary_node_id': unary_node_ids,
        'unary_node_time': unary_node_times,
        'unary_sts_error': unary_sts_e,
        'unary_ets_error': unary_ets_e,
        'unary_dist_from_sample_centroid0': unary_dist_from_sample_centroid0,
        'unary_sts_dist_from_sample_centroid': unary_sts_dist_from_sample_centroid,
        'unary_ets_dist_from_sample_centroid': unary_ets_dist_from_sample_centroid,
        'added_span': total_added_span,
        'wrongly_added_span': wrong_added_span,
        'sts_span': sts_spans,
        'ets_span': ets_spans
    })

    child_df = pd.DataFrame({
        'child_node_id': child_node_ids,
        'child_node_time': child_node_times,
        'child_sts_error': child_sts_e,
        'child_ets_error': child_ets_e,
        'child_dist_from_sample_centroid0': child_dist_from_sample_centroid0,
        'child_sts_dist_from_sample_centroid': child_sts_dist_from_sample_centroid,
        'child_ets_dist_from_sample_centroid': child_ets_dist_from_sample_centroid
    })

    parent_df = pd.DataFrame({
        'parent_node_id': parent_node_ids,
        'parent_node_time': parent_node_times,
        'parent_sts_error': parent_sts_e,
        'parent_ets_error': parent_ets_e,
        'parent_dist_from_sample_centroid0': parent_dist_from_sample_centroid0,
        'parent_sts_dist_from_sample_centroid': parent_sts_dist_from_sample_centroid,
        'parent_ets_dist_from_sample_centroid': parent_ets_dist_from_sample_centroid

    })

    static_df = pd.DataFrame({
        'sigma': [sigma],
        'rep': [rep],
        'sts_num_trees': [sts_num_trees],
        'ets_num_trees': [ets_num_trees], 
        'sts_num_edges': [sts_num_edges],
        'ets_num_edges': [ets_num_edges]
    })


    unary_df.to_csv(f"{out_prefix}_unary_results.csv",
                      mode='a', header=True, index=False)
    
    child_df.to_csv(f"{out_prefix}_child_results.csv",
                      mode='a', header=True, index=False)
    
    parent_df.to_csv(f"{out_prefix}_parent_results.csv",
                      mode='a', header=True, index=False)
    
    static_df.to_csv(f"{out_prefix}_static_info.csv",
                      mode='a', header=True, index=False)

    
    node_stats_df.to_csv(f"{out_prefix}_node_stats.csv",
                      mode='a', header=True, index=False)

    return unary_df, child_df, parent_df, static_df, node_stats_df


In [ ]:
ts_path = 'tree-S0.2-R0.trees'
out_prefix = 'S0.2-R0'
sigma = 0.2
rep = 0
getAccOut(ts_path, out_prefix, sigma, rep)

In [ ]:

# Create a new process
process = Process()

def func(num):
    print("hello world")
    print(num)

process = Process(target=func, args=("yooo",))

process.start()

process.is_alive()

In [ ]:
def do_something():
    print("I'm going to sleep")
    time.sleep(1)
    print("I'm awake") 

process_1 = Process(target=do_something)
process_2 = Process(target=do_something)

In [ ]:
%%time

# Create new child process (Cannot run a process more than once)
new_process_1 = Process(target=do_something)
new_process_2 = Process(target=do_something)

# Starts both processes
new_process_1.start()
new_process_2.start()

new_process_1.join()
new_process_2.join()

In [ ]:
if process_1.is_alive():
    process_1.terminate() # You can also use process.kill()

if process_2.is_alive():
    process_2.terminate() # Y